# Day 046 — Exercise 3: Rolling Windows

**What you'll build:** `rolling_mean(series, window, min_periods=1) -> pd.Series` — smooth a noisy series with a moving average. And `rolling_stats(series, window) -> pd.DataFrame` — compute rolling mean, std, min, and max in one DataFrame.

**Why it matters:** Raw time series is noisy. A 7-day rolling mean strips that noise and shows the underlying trend. Rolling std reveals volatility spikes. Together they are the most-used tools for pattern detection in time series.

## Provided: Setup + parse_time_series + resample_series

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)


def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }

## Your Implementation

In [ ]:
def rolling_mean(series: pd.Series, window: int,
                 min_periods: int = 1) -> pd.Series:
    """
    Compute rolling mean with min_periods=1 so the first values aren't NaN.

    Args:
        series:      datetime-indexed Series
        window:      look-back window (number of periods)
        min_periods: minimum observations needed (default 1 — no leading NaN)
    Returns:
        pd.Series of rolling means, same length as input
    """
    # TODO: return series.rolling(window=window, min_periods=min_periods).mean()
    pass


def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    """
    Compute rolling mean, std, min, max in one DataFrame.

    Uses min_periods=1 so no leading NaN rows.
    Returns DataFrame with columns: mean, std, min, max.
    """
    # TODO: r = series.rolling(window=window, min_periods=1)
    # TODO: return pd.DataFrame({
    #     'mean': r.mean(), 'std': r.std(),
    #     'min':  r.min(),  'max': r.max(),
    # })
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df_raw = make_sample_ts(30)
    series = parse_time_series(df_raw, 'date')['value']

    # Check 1: rolling_mean returns same-length Series
    try:
        assert 'rolling_mean' in globals()
        rm = rolling_mean(series, window=7)
        assert isinstance(rm, pd.Series), \
            f'expected Series, got {type(rm).__name__}'
        assert len(rm) == len(series), \
            f'rolling_mean must preserve length: {len(rm)} vs {len(series)}'
        passed += 1; print('\u2705 Check 1: rolling_mean returns same-length Series')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: no NaN in first value (min_periods=1)
    try:
        assert not pd.isna(rm.iloc[0]), \
            'first rolling_mean value should not be NaN (min_periods=1)'
        passed += 1; print('\u2705 Check 2: no leading NaN — min_periods=1 working')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: rolling_stats returns DataFrame with correct shape
    try:
        assert 'rolling_stats' in globals()
        rs = rolling_stats(series, window=7)
        assert isinstance(rs, pd.DataFrame), \
            f'expected DataFrame, got {type(rs).__name__}'
        assert len(rs) == len(series), \
            f'rolling_stats must preserve length'
        passed += 1; print('\u2705 Check 3: rolling_stats returns same-length DataFrame')
    except Exception as e:
        print(f'\u274c Check 3: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 4: has mean, std, min, max columns
    try:
        for col in ('mean', 'std', 'min', 'max'):
            assert col in rs.columns, f'missing column: {col}'
        passed += 1; print('\u2705 Check 4: rolling_stats has mean, std, min, max')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: rolling_stats mean == rolling_mean for same window
    try:
        rm2 = rolling_mean(series, window=7)
        diff = (rs['mean'] - rm2).abs().max()
        assert diff < 1e-9, f'rolling_stats mean != rolling_mean (max diff={diff})'
        passed += 1; print('\u2705 Check 5: rolling_stats[mean] == rolling_mean')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def rolling_mean(series: pd.Series, window: int, min_periods: int = 1) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).mean()

def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    r = series.rolling(window=window, min_periods=1)
    return pd.DataFrame({
        'mean': r.mean(),
        'std':  r.std(),
        'min':  r.min(),
        'max':  r.max(),
    })
```

</details>